In [ ]:
!pip install pandas numpy matplotlib seaborn sqlalchemy openpyxl

In [ ]:
import pandas as pd

df = pd.read_csv('/content/customer_shopping_behavior.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()


In [ ]:
df.describe(include = 'all')

In [ ]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


In [ ]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [ ]:
!ls /content

sample_data


In [ ]:
import os

for file in os.listdir("/content"):
    print(file)

.config
sample_data


In [ ]:
!wget https://raw.githubusercontent.com/amlanmohanty1/customer-trends-data-analysis-SQL-Python-PowerBI/main/customer_shopping_behavior.csv

--2026-08-11 09:33:42--  https://raw.githubusercontent.com/amlanmohanty1/customer-trends-data-analysis-SQL-Python-PowerBI/main/customer_shopping_behavior.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 416506 (407K) [text/plain]
Saving to: ‘customer_shopping_behavior.csv’

customer_shopping_b 100%[===================>] 406.74K  --.-KB/s    in 0.04s   

2026-08-11 09:33:43 (11.0 MB/s) - ‘customer_shopping_behavior.csv’ saved [416506/416506]



In [ ]:
!ls

customer_shopping_behavior.csv	sample_data


In [ ]:
import pandas as pd

df = pd.read_csv('customer_shopping_behavior.csv')
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [ ]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df = df.rename(columns = {'purchase_amount_(usd)' : 'purchase_amount'})

In [ ]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [ ]:
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)

In [ ]:
df[['age' , 'age_group']].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


In [ ]:
frequency_mapping = { 'Fortnightly': 14, 'Weekly' : 7, 'Monthly' : 30, 'Quarterly' : 90, 'Bi-Weekly' : 14, 'Annually' : 365, 'Every 3 Months' : 90}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [ ]:
df[['purchase_frequency_days', 'frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [ ]:
df[['discount_applied' , 'promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [ ]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [ ]:
df = df.drop('promo_code_used', axis = 1)

In [ ]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [ ]:
pip install psycopg2-binary sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 86.2 MB/s eta 0:00:00


In [ ]:
from sqlalchemy import create_engine
from getpass import getpass

username = "postgres.astfxwkcixjyyjylrrrm"
password = getpass("Enter your Supabase database password: ")

host = "aws-0-ap-southeast-2.pooler.supabase.com"
port = "5432"
database = "postgres"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

Enter your Supabase database password: ··········


In [ ]:
with engine.connect() as connection:
    result = connection.exec_driver_sql("SELECT version();")
    print(result.fetchone())

('PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit',)


In [ ]:
table_name = "customer"
df.to_sql(table_name, engine, if_exists="replace", index=False)
print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")

Data successfully loaded into table 'customer' in database 'postgres'.
